[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VaishnaviJagtap18/42-days-aiml-challenge/blob/main/week4_deep_learning/day31_backprop_optimizers/day31_notebook.ipynb)

# Day 31 / 42: Backpropagation and Optimizers
### #42DaysOfML | Week 4: Deep Learning

---

## What You'll Learn
- Backpropagation step by step — forward pass, loss, backward pass
- The chain rule in actual code, not abstract math
- Implement backprop manually for a 2-layer network
- SGD vs Momentum vs RMSProp vs Adam — what each fixes
- Build each optimizer from scratch and compare on the same task
- Learning rate schedulers: warmup, cosine, step decay
- Production problem: learning rate choice breaking training

---

In [ ]:
!pip install torch matplotlib numpy --quiet

## The Concept

### Backpropagation

A neural network makes predictions through a **forward pass**: data flows through layers, each applying a transformation (linear + activation). The output is compared to the true label using a **loss function**. The loss is a single number measuring how wrong the prediction is.

To improve, you need to know: *how does changing each weight affect the loss?* That's the gradient: `dL/dW` for each weight matrix W.

Backpropagation computes these gradients efficiently using the **chain rule** of calculus:

```
dL/dW1 = dL/dA2 * dA2/dZ2 * dZ2/dA1 * dA1/dZ1 * dZ1/dW1
```

You compute from the output layer backward, reusing intermediate results. One forward pass + one backward pass gives you the gradient for every weight in the network.

### Optimizers

Once you have gradients, you update weights. The simplest rule is SGD:
```
W = W - lr * dL/dW
```

But SGD has problems:
- Oscillates in narrow loss valleys (high curvature in one direction, low in another)
- Slows down in flat regions (small gradients -> tiny steps)
- Uses the same learning rate for every parameter

Modern optimizers fix these one by one:
- **Momentum**: adds a velocity term. Past gradients influence current update. Reduces oscillation.
- **RMSProp**: adapts learning rate per parameter based on recent gradient magnitude. Large gradients -> smaller steps, small gradients -> larger steps.
- **Adam**: combines Momentum (first moment) + RMSProp (second moment). Corrects for bias in early training. The default optimizer for almost every neural network today.
- **AdamW**: Adam + proper weight decay (decoupled from gradient update). Default for training Transformers.

In [ ]:
# ============================================================
# SECTION 1: Backpropagation From Scratch
# Build a 2-layer neural network and implement backward pass manually
# Then verify every gradient against PyTorch autograd
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# --- Activation functions and their derivatives ---
def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    """Gradient of ReLU: 1 where x > 0, 0 otherwise."""
    return (x > 0).astype(float)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

def bce_loss(y_pred, y_true):
    """Binary cross-entropy loss."""
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


class TwoLayerNet:
    """
    Network: Input -> Linear -> ReLU -> Linear -> Sigmoid -> Output
    Architecture: input_size -> hidden_size -> 1
    """
    def __init__(self, input_size, hidden_size):
        # He initialisation for ReLU layers: std = sqrt(2/fan_in)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, 1) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, 1))
        
        # Cache intermediate values needed for backward pass
        self.cache = {}
    
    def forward(self, X):
        """
        Forward pass — compute output and cache everything needed for backward.
        X: (batch_size, input_size)
        """
        # Layer 1: Linear + ReLU
        Z1 = X @ self.W1 + self.b1     # pre-activation: (batch, hidden)
        A1 = relu(Z1)                   # post-activation: (batch, hidden)
        
        # Layer 2: Linear + Sigmoid
        Z2 = A1 @ self.W2 + self.b2    # pre-activation: (batch, 1)
        A2 = sigmoid(Z2)               # output probability: (batch, 1)
        
        # Cache everything needed for backward
        self.cache = {'X': X, 'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}
        return A2
    
    def backward(self, y_true):
        """
        Backward pass — compute gradients for all parameters using chain rule.
        
        Chain rule from output to W1:
        dL/dW1 = dL/dA2 * dA2/dZ2 * dZ2/dA1 * dA1/dZ1 * dZ1/dW1
        """
        X  = self.cache['X']
        Z1 = self.cache['Z1']
        A1 = self.cache['A1']
        Z2 = self.cache['Z2']
        A2 = self.cache['A2']
        m  = X.shape[0]  # batch size
        
        # --- Gradient of loss w.r.t. output ---
        # For BCE loss + sigmoid output, this simplifies to: (A2 - y) / m
        dA2 = (A2 - y_true) / m          # (batch, 1)
        
        # --- Gradients for Layer 2 ---
        # dL/dZ2 = dL/dA2 * dA2/dZ2 = dA2 * sigmoid'(Z2)
        dZ2 = dA2 * sigmoid_derivative(Z2)   # (batch, 1)
        
        # dL/dW2 = A1^T * dZ2    (chain rule through Z2 = A1 @ W2 + b2)
        dW2 = A1.T @ dZ2                      # (hidden, 1)
        db2 = np.sum(dZ2, axis=0, keepdims=True)  # (1, 1)
        
        # --- Gradients for Layer 1 ---
        # dL/dA1 = dZ2 @ W2^T    (backprop through W2)
        dA1 = dZ2 @ self.W2.T                # (batch, hidden)
        
        # dL/dZ1 = dA1 * ReLU'(Z1)  (backprop through ReLU)
        dZ1 = dA1 * relu_derivative(Z1)      # (batch, hidden)
        
        # dL/dW1 = X^T * dZ1
        dW1 = X.T @ dZ1                      # (input, hidden)
        db1 = np.sum(dZ1, axis=0, keepdims=True)  # (1, hidden)
        
        return {'dW1': dW1, 'db1': db1, 'dW2': dW2, 'db2': db2}


# --- Generate a simple binary classification dataset ---
from sklearn.datasets import make_moons
X_data, y_data = make_moons(n_samples=200, noise=0.2, random_state=42)
y_data = y_data.reshape(-1, 1).astype(float)

# Simple normalisation
X_data = (X_data - X_data.mean(axis=0)) / X_data.std(axis=0)

net = TwoLayerNet(input_size=2, hidden_size=16)

# One forward + backward pass
output = net.forward(X_data)
loss = bce_loss(output, y_data)
grads = net.backward(y_data)

print("Forward + Backward Pass")
print("=" * 45)
print(f"Input shape:        {X_data.shape}")
print(f"Output shape:       {output.shape}")
print(f"Loss:               {loss:.4f}")
print(f"\nGradient shapes:")
for name, g in grads.items():
    print(f"  {name}: {g.shape}  (same shape as the parameter)")

print(f"\nGradient magnitudes (||grad||):")
for name, g in grads.items():
    print(f"  ||{name}||: {np.linalg.norm(g):.4f}")

In [ ]:
# ============================================================
# SECTION 2: Verify Manual Gradients Against PyTorch Autograd
# If they match, our backprop is correct
# ============================================================
import torch
import torch.nn as nn

torch.manual_seed(42)

# Recreate same network in PyTorch with identical weights
X_torch = torch.tensor(X_data, dtype=torch.float32)
y_torch = torch.tensor(y_data, dtype=torch.float32)

# Use our manually initialised weights for exact comparison
W1_t = torch.tensor(net.W1.T, dtype=torch.float32, requires_grad=True)  # PyTorch: (out, in)
b1_t = torch.tensor(net.b1.flatten(), dtype=torch.float32, requires_grad=True)
W2_t = torch.tensor(net.W2.T, dtype=torch.float32, requires_grad=True)
b2_t = torch.tensor(net.b2.flatten(), dtype=torch.float32, requires_grad=True)

# Forward pass in PyTorch
Z1_t = X_torch @ W1_t.T + b1_t
A1_t = torch.relu(Z1_t)
Z2_t = A1_t @ W2_t.T + b2_t
A2_t = torch.sigmoid(Z2_t)

loss_t = nn.BCELoss()(A2_t, y_torch)
loss_t.backward()

# Compare
print("Gradient Verification: Manual vs PyTorch Autograd")
print("=" * 55)
comparisons = [
    ('dW1', grads['dW1'], W1_t.grad.numpy().T),
    ('db1', grads['db1'].flatten(), b1_t.grad.numpy()),
    ('dW2', grads['dW2'], W2_t.grad.numpy().T),
    ('db2', grads['db2'].flatten(), b2_t.grad.numpy()),
]

all_match = True
for name, manual, auto in comparisons:
    max_diff = np.max(np.abs(manual - auto))
    match = max_diff < 1e-5
    all_match = all_match and match
    status = 'MATCH' if match else 'MISMATCH'
    print(f"  {name}: max diff = {max_diff:.2e}  [{status}]")

print(f"\nAll gradients match PyTorch: {all_match}")
print(f"\nThis confirms our manual backprop is mathematically correct.")
print(f"PyTorch autograd does exactly the same chain rule computation,")
print(f"but tracks the computational graph automatically.")

In [ ]:
# ============================================================
# SECTION 3: Build SGD, Momentum, RMSProp, and Adam From Scratch
# Run all four on the same network and same data
# ============================================================

class SGD:
    """Vanilla stochastic gradient descent."""
    def __init__(self, lr=0.01):
        self.lr = lr
    
    def update(self, params, grads):
        for key in params:
            params[key] -= self.lr * grads['d' + key]
        return params


class SGDMomentum:
    """
    SGD with momentum.
    Maintains a velocity vector that accumulates past gradients.
    v = beta * v - lr * grad
    W = W + v
    """
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.v = {}  # velocity
    
    def update(self, params, grads):
        for key in params:
            if key not in self.v:
                self.v[key] = np.zeros_like(params[key])
            grad_key = 'd' + key
            # Momentum: accumulate velocity
            self.v[key] = self.beta * self.v[key] - self.lr * grads[grad_key]
            params[key] += self.v[key]
        return params


class RMSProp:
    """
    RMSProp: adapts learning rate per parameter.
    Maintains exponential moving average of squared gradients.
    s = beta * s + (1-beta) * grad^2
    W = W - lr * grad / (sqrt(s) + eps)
    """
    def __init__(self, lr=0.001, beta=0.999, eps=1e-8):
        self.lr = lr
        self.beta = beta
        self.eps = eps
        self.s = {}  # second moment
    
    def update(self, params, grads):
        for key in params:
            if key not in self.s:
                self.s[key] = np.zeros_like(params[key])
            grad_key = 'd' + key
            g = grads[grad_key]
            # Running average of squared gradients
            self.s[key] = self.beta * self.s[key] + (1 - self.beta) * g**2
            # Adaptive step: divide by sqrt of average squared gradient
            params[key] -= self.lr * g / (np.sqrt(self.s[key]) + self.eps)
        return params


class Adam:
    """
    Adam: Adaptive Moment Estimation.
    Combines Momentum (first moment m) + RMSProp (second moment v).
    Adds bias correction because m and v start at zero.
    
    m = beta1 * m + (1-beta1) * grad           <- first moment (like momentum)
    v = beta2 * v + (1-beta2) * grad^2          <- second moment (like RMSProp)
    m_hat = m / (1 - beta1^t)                   <- bias correction
    v_hat = v / (1 - beta2^t)                   <- bias correction
    W = W - lr * m_hat / (sqrt(v_hat) + eps)
    """
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = {}   # first moment
        self.v = {}   # second moment
        self.t = 0    # time step for bias correction
    
    def update(self, params, grads):
        self.t += 1
        for key in params:
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            
            grad_key = 'd' + key
            g = grads[grad_key]
            
            # Update biased moments
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * g
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * g**2
            
            # Bias correction — critical in early training
            m_hat = self.m[key] / (1 - self.beta1**self.t)
            v_hat = self.v[key] / (1 - self.beta2**self.t)
            
            params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return params


print("All 4 optimizers defined.")
print("SGD:      W = W - lr * grad")
print("Momentum: v = beta*v - lr*grad; W = W + v")
print("RMSProp:  s = beta*s + (1-beta)*grad^2; W = W - lr*grad/sqrt(s)")
print("Adam:     combines Momentum + RMSProp with bias correction")

In [ ]:
# ============================================================
# SECTION 4: Compare All Optimizers on the Same Task
# Same network, same data, same initialisation
# ============================================================

def train_network(optimizer_class, optimizer_kwargs, epochs=200, batch_size=32, seed=42):
    """Train TwoLayerNet with given optimizer and return loss history."""
    np.random.seed(seed)
    net = TwoLayerNet(input_size=2, hidden_size=32)
    opt = optimizer_class(**optimizer_kwargs)
    losses = []
    accs = []
    
    n_samples = len(X_data)
    
    for epoch in range(epochs):
        # Mini-batch SGD
        indices = np.random.permutation(n_samples)
        epoch_loss = 0
        n_batches = 0
        
        for start in range(0, n_samples, batch_size):
            batch_idx = indices[start:start+batch_size]
            X_batch = X_data[batch_idx]
            y_batch = y_data[batch_idx]
            
            # Forward
            output = net.forward(X_batch)
            loss = bce_loss(output, y_batch)
            epoch_loss += loss
            n_batches += 1
            
            # Backward
            grads_net = net.backward(y_batch)
            
            # Update
            params = {'W1': net.W1, 'b1': net.b1, 'W2': net.W2, 'b2': net.b2}
            updated = opt.update(params, grads_net)
            net.W1, net.b1, net.W2, net.b2 = updated['W1'], updated['b1'], updated['W2'], updated['b2']
        
        losses.append(epoch_loss / n_batches)
        
        # Compute accuracy on full dataset
        preds = net.forward(X_data) >= 0.5
        acc = (preds == y_data).mean()
        accs.append(acc)
    
    final_pred = net.forward(X_data) >= 0.5
    final_acc = (final_pred == y_data).mean() * 100
    return losses, accs, net, final_acc


# Train with all four optimizers
print("Training with 4 optimizers on make_moons dataset...")
print("Same architecture, same data, same random seed.")
print("=" * 50)

optimizer_configs = [
    ('SGD',           SGD,          {'lr': 0.05},                       '#F44336'),
    ('SGD+Momentum',  SGDMomentum,  {'lr': 0.05, 'beta': 0.9},          '#FF9800'),
    ('RMSProp',       RMSProp,      {'lr': 0.001, 'beta': 0.999},        '#2196F3'),
    ('Adam',          Adam,         {'lr': 0.001, 'beta1': 0.9, 'beta2': 0.999}, '#4CAF50'),
]

results = {}
for name, cls, kwargs, color in optimizer_configs:
    losses, accs, trained_net, final_acc = train_network(cls, kwargs, epochs=200)
    results[name] = {'losses': losses, 'accs': accs, 'net': trained_net, 'color': color}
    print(f"  {name:15s}: Final Accuracy = {final_acc:.1f}%  |  Final Loss = {losses[-1]:.4f}")

# Plot convergence
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for name, data in results.items():
    axes[0].plot(data['losses'], label=name, color=data['color'], linewidth=2)
    axes[1].plot([a*100 for a in data['accs']], label=name, color=data['color'], linewidth=2)

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss: SGD vs Momentum vs RMSProp vs Adam', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(bottom=0)

axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training Accuracy: SGD vs Momentum vs RMSProp vs Adam', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Visualise Decision Boundaries for Each Optimizer
# ============================================================

def plot_decision_boundary(net, X, y, ax, title):
    """Plot the decision boundary learned by a trained network."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = net.forward(grid).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='RdBu', levels=20)
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y.ravel(), cmap='RdBu',
                        edgecolors='black', linewidth=0.5, s=40, zorder=2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for (name, data), ax in zip(results.items(), axes.flat):
    final_acc = (data['net'].forward(X_data) >= 0.5 == y_data).mean() * 100
    plot_decision_boundary(data['net'], X_data, y_data, ax,
                           f"{name}\nAccuracy: {final_acc:.1f}%")

plt.suptitle('Decision Boundaries After 200 Epochs\n(Same network, same data, different optimizer)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SECTION 5: Why Adam's Bias Correction Matters
# Show what happens without it in early training
# ============================================================

# Simulate Adam with and without bias correction for first 20 steps
beta1, beta2, lr, eps = 0.9, 0.999, 0.001, 1e-8
grad_val = 1.0  # constant gradient for simplicity

m, v = 0.0, 0.0
steps = 50

effective_lr_corrected = []
effective_lr_uncorrected = []
step_sizes = []

for t in range(1, steps + 1):
    m = beta1 * m + (1 - beta1) * grad_val
    v = beta2 * v + (1 - beta2) * grad_val**2
    
    # With bias correction
    m_hat = m / (1 - beta1**t)
    v_hat = v / (1 - beta2**t)
    step_corrected = lr * m_hat / (np.sqrt(v_hat) + eps)
    
    # Without bias correction
    step_uncorrected = lr * m / (np.sqrt(v) + eps)
    
    effective_lr_corrected.append(step_corrected)
    effective_lr_uncorrected.append(step_uncorrected)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, steps+1), effective_lr_corrected, label='Adam WITH bias correction', 
        color='#4CAF50', linewidth=2.5)
ax.plot(range(1, steps+1), effective_lr_uncorrected, label='Adam WITHOUT bias correction', 
        color='#F44336', linewidth=2.5, linestyle='--')
ax.axhline(y=lr, color='gray', linestyle=':', linewidth=1.5, label=f'Target LR = {lr}')
ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Effective Step Size', fontsize=12)
ax.set_title('Adam Bias Correction: Effect in Early Training\n'
             'Without correction, early steps are nearly zero — network barely moves',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At step 1:")
print(f"  With bias correction:    step size = {effective_lr_corrected[0]:.6f}")
print(f"  Without bias correction: step size = {effective_lr_uncorrected[0]:.6f}")
print(f"\nAt step 50:")
print(f"  With bias correction:    step size = {effective_lr_corrected[49]:.6f}")
print(f"  Without bias correction: step size = {effective_lr_uncorrected[49]:.6f}")
print(f"\nWithout bias correction, first few steps are 10-100x smaller than intended.")
print(f"The network barely learns anything in the first few epochs.")
print(f"Bias correction fixes this: step sizes are near the target LR from step 1.")

In [ ]:
# ============================================================
# SECTION 6: Learning Rate Schedulers
# Visualise and compare: step decay, cosine annealing, warmup+cosine
# ============================================================
import torch
import torch.optim as optim

def get_lr_schedule(scheduler_name, total_steps=200, warmup_steps=20):
    """Simulate LR schedule without actually training."""
    # Dummy model and optimizer
    dummy_param = [torch.nn.Parameter(torch.randn(1))]
    optimizer = optim.Adam(dummy_param, lr=0.001)
    
    if scheduler_name == 'step':
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.5)
    elif scheduler_name == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)
    elif scheduler_name == 'cosine_warmup':
        def lr_lambda(step):
            if step < warmup_steps:
                return step / warmup_steps  # linear warmup
            # Cosine decay after warmup
            progress = (step - warmup_steps) / (total_steps - warmup_steps)
            return 0.5 * (1 + np.cos(np.pi * progress))
        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    elif scheduler_name == 'exponential':
        scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.97)
    elif scheduler_name == 'constant':
        return [0.001] * total_steps
    
    lrs = []
    for step in range(total_steps):
        lrs.append(optimizer.param_groups[0]['lr'])
        scheduler.step()
    return lrs


schedules = {
    'Constant LR':       ('constant',      '#9E9E9E'),
    'Step Decay':        ('step',          '#F44336'),
    'Cosine Annealing':  ('cosine',        '#2196F3'),
    'Warmup + Cosine':   ('cosine_warmup', '#4CAF50'),
    'Exponential Decay': ('exponential',   '#FF9800'),
}

fig, ax = plt.subplots(figsize=(13, 6))
for label, (name, color) in schedules.items():
    lrs = get_lr_schedule(name)
    ax.plot(lrs, label=label, color=color, linewidth=2.5)

ax.axvline(x=20, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Warmup end (step 20)')
ax.set_xlabel('Training Step (Epoch)', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Learning Rate Schedulers Compared\n(all starting from lr=0.001)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("When to use each scheduler:")
print()
print("  Constant LR:      Simple baseline. Works for short training runs.")
print("  Step Decay:       Drop LR at fixed intervals. Common in ResNet papers.")
print("                    Requires knowing when to drop — fragile.")
print("  Cosine Annealing: Smooth decay to near-zero. Good for CNNs and RNNs.")
print("                    Often used with restarts (SGDR).")
print("  Warmup + Cosine:  Standard for Transformer training. Start with low LR")
print("                    to stabilise Adam's biased moments, then decay smoothly.")
print("                    Used in BERT, GPT, T5, LLaMA.")
print("  Exponential:      Fast decay. Can be too aggressive for long training.")

In [ ]:
# ============================================================
# SECTION 7: Effect of Learning Rate on Training Dynamics
# Demonstrate: too high LR diverges, too low never converges
# ============================================================

learning_rates = [0.1, 0.01, 0.001, 0.0001, 0.00001]
colors = ['#F44336', '#FF9800', '#4CAF50', '#2196F3', '#9C27B0']
labels = ['0.1 (too high — diverges)', '0.01 (fast)', '0.001 (sweet spot)', 
          '0.0001 (slow)', '0.00001 (too low — barely moves)']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

all_losses = {}
for lr, color, label in zip(learning_rates, colors, labels):
    np.random.seed(42)
    net_lr = TwoLayerNet(input_size=2, hidden_size=32)
    opt = Adam(lr=lr)
    losses_lr = []
    
    for epoch in range(100):
        output = net_lr.forward(X_data)
        loss = bce_loss(output, y_data)
        if np.isnan(loss) or loss > 10:  # diverged
            losses_lr.extend([float('nan')] * (100 - epoch))
            break
        losses_lr.append(loss)
        grads_lr = net_lr.backward(y_data)
        params = {'W1': net_lr.W1, 'b1': net_lr.b1, 'W2': net_lr.W2, 'b2': net_lr.b2}
        opt.update(params, grads_lr)
        net_lr.W1, net_lr.b1, net_lr.W2, net_lr.b2 = params['W1'], params['b1'], params['W2'], params['b2']
    
    # Filter NaN for plotting
    valid_losses = [l for l in losses_lr if not np.isnan(l)]
    axes[0].plot(range(len(valid_losses)), valid_losses, 
                 label=f'lr={lr}', color=color, linewidth=2)
    all_losses[lr] = losses_lr

axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Effect of Learning Rate on Loss\n(Adam optimizer, same network)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1.5)

# Show loss at epoch 100 for each LR
final_losses = []
for lr in learning_rates:
    valid = [l for l in all_losses[lr] if not np.isnan(l)]
    final_losses.append(valid[-1] if valid else 2.0)

bar_colors = ['#F44336' if fl > 1.0 else '#4CAF50' if fl < 0.3 else '#FF9800' for fl in final_losses]
bars = axes[1].bar([str(lr) for lr in learning_rates], final_losses, 
                    color=bar_colors, edgecolor='black', alpha=0.85)
for bar, fl in zip(bars, final_losses):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{fl:.3f}', ha='center', fontsize=11, fontweight='bold')

axes[1].set_xlabel('Learning Rate', fontsize=12)
axes[1].set_ylabel('Final Loss (epoch 100)', fontsize=12)
axes[1].set_title('Final Loss vs Learning Rate\n(green = good, red = bad)', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Key observations:")
print("  lr=0.1:     Diverges — steps too large, overshoots the minimum")
print("  lr=0.01:    Converges fast but may overshoot in the last phase")
print("  lr=0.001:   Default Adam — converges reliably across most tasks")
print("  lr=0.0001:  Converges slowly — usable but wasteful")
print("  lr=0.00001: Barely moves — essentially no learning in 100 epochs")

## Real World Problem: Learning Rate Killing a Production Training Run

An ML team at a startup spends 3 weeks preparing a dataset and starts training a large image classifier on a cloud GPU instance. The loss drops for 2 hours then suddenly spikes and never recovers. The run costs $800 in compute. They restart with the same settings. It happens again.

**What happened:** The learning rate was too high for the batch size they were using. They increased batch size from 32 to 256 to use the GPU more efficiently but didn't scale the learning rate. With batch size 256, each gradient estimate is smoother (less noise), which means the effective learning rate is higher than intended. The training became unstable.

**The linear scaling rule (from Facebook's 2017 large-batch training paper):**

When you multiply batch size by k, multiply learning rate by k as well. Going from batch=32 to batch=256 (8x increase) should multiply lr by 8x.

Combined with linear warmup for the first 5 epochs: start from lr/k, linearly increase to lr. This prevents the instability from large gradient updates at the start.

**Three other learning rate bugs that show up in production:**

1. **Not resetting the optimizer state when resuming training from a checkpoint.** Adam's first and second moments are stale and the effective step size is wrong for the first 100-200 steps.

2. **Using the same learning rate for the pretrained backbone and the new head in fine-tuning.** The backbone needs 10-100x lower LR to avoid catastrophic forgetting.

3. **Setting a scheduler that reduces LR on plateau, but evaluating on a noisy validation metric.** Random noise in validation causes premature LR reduction and the model gets stuck before it should.

## Interview Corner: MNC-Level Questions

---

**Q1: Explain backpropagation in plain terms without using the phrase "chain rule".**

*What they're testing:* Whether you genuinely understand it or just memorised the term.

*Answer direction:* During training, the network makes a prediction and the loss tells us how wrong it was. To improve, we need to know how each weight contributed to that error. Backprop works backward from the loss: starting at the output layer, it computes how much the loss would change if you nudged each weight by a tiny amount. Then it passes that signal to the previous layer. Each layer receives "here's how sensitive the loss is to my output" from the layer ahead of it, and uses that to compute "here's how sensitive the loss is to my input and my weights." This cascades all the way to the first layer. The result is a gradient for every weight — a direction and magnitude saying "move this weight this much to reduce the loss."

---

**Q2: Why does Adam use two separate moment estimates instead of just one?**

*What they're testing:* Deep understanding of Adam's design.

*Answer direction:* The first moment (m, like momentum) tracks the direction of recent gradients. This helps the optimizer maintain momentum through small gradient regions and smooth out oscillations. But if we only had this, parameters with consistently large gradients would still take huge steps. The second moment (v, like RMSProp) tracks the magnitude of recent gradients per parameter. Dividing by sqrt(v) normalises the step size: parameters with large gradients get smaller steps, parameters with small gradients get larger steps. Together they solve both problems momentum solves (direction and smoothing) and RMSProp solves (scale adaptation) simultaneously.

---

**Q3: Your model trains well with batch size 32 but diverges with batch size 512. Why?**

*What they're testing:* Understanding of batch size and learning rate interaction.

*Answer direction:* Larger batches give lower-variance gradient estimates, which means each update is more "confident" and can have a larger effective impact. With the same learning rate, larger batches take larger effective steps in parameter space. This can overshoot the loss minimum, causing divergence. Fix: scale the learning rate linearly with batch size (the linear scaling rule from Goyal et al. 2017). Going from 32 to 512 (16x) should increase lr by 16x, combined with a linear warmup for the first 5 epochs. In practice many teams use gradient accumulation instead: simulate large batches by accumulating gradients over multiple small batches before doing one weight update. Same effective batch size, no LR change needed.

---

**Q4: What is gradient clipping, when should you use it, and what value should you clip to?**

*What they're testing:* Training stability knowledge.

*Answer direction:* Gradient clipping caps the L2 norm of the gradient vector. If ||grad|| > threshold, scale all gradients by threshold/||grad||. This prevents a single catastrophically large gradient update from throwing the model into a bad region. Use it for: (a) RNNs/LSTMs which suffer from exploding gradients, (b) Transformer training where occasional large gradient spikes occur, (c) any task where you see sudden loss spikes during training. Common values: 1.0 for most Transformers (BERT, GPT use this), 5.0 for RNNs. The value should be roughly 90th-95th percentile of gradient norms during a healthy early training run — large enough not to trigger constantly, small enough to catch true spikes.

---

**Q5: What is the difference between AdamW and Adam, and why does it matter for fine-tuning LLMs?**

*What they're testing:* Current awareness of best practices.

*Answer direction:* Adam applies L2 regularisation (weight decay) by adding lambda*W to the gradient before the adaptive scaling: `g = g + lambda*W`. This means the weight decay is divided by the second moment estimate, so parameters with large gradients get less regularisation than parameters with small gradients. This is inconsistent with what L2 regularisation is supposed to do. AdamW decouples weight decay from the gradient update: it applies the decay directly to the weight after the Adam update: `W = (1 - lambda) * W - lr * m_hat / (sqrt(v_hat) + eps)`. This makes weight decay consistent across all parameters regardless of gradient magnitude. In fine-tuning LLMs this matters because some layers (like embeddings) have very different gradient magnitudes than attention layers, and inconsistent regularisation causes uneven training.


## ML Spotlight

**Muon Optimizer (2024)**

Muon (Momentum + Orthogonalization via Newton-Schulz) is a new optimizer from Keller Jordan that has shown strong results for training neural networks at scale. It applies orthogonalisation to the gradient before the update, ensuring each layer's weight update is in a direction that doesn't interfere with other layers. Early results on language models show 1.5-2x faster convergence than AdamW at the same step count.

Muon represents the active research direction in optimizers: rather than just adapting learning rates per parameter (Adam), researchers are now looking at the geometry of the loss landscape at a higher level.

For production today: AdamW remains the default for most tasks. But Muon and similar approaches (Shampoo, SOAP) are worth tracking as they mature.

Paper and code: https://github.com/KellerJordan/modded-nanogpt (includes Muon implementation)

Blog post: https://kellerjordan.github.io/posts/muon

## Practice Exercise

1. Add L2 weight decay to the Adam optimizer class above:
```python
# In Adam.update(), before the moment updates:
g = grads[grad_key] + self.weight_decay * params[key]
```
This is standard Adam with L2 regularisation. Train both Adam and this AdamW-style version. Does weight decay help on make_moons?

2. Implement a learning rate warmup + cosine decay schedule as a wrapper around the Adam class. Track effective LR at each step alongside loss.

3. Implement gradient clipping in the TwoLayerNet backward pass:
```python
# Compute global gradient norm
total_norm = np.sqrt(sum(np.sum(g**2) for g in grads.values()))
clip_coef = min(1.0, max_norm / (total_norm + 1e-6))
# Scale all gradients
grads = {k: v * clip_coef for k, v in grads.items()}
```
Train with max_norm = [0.1, 0.5, 1.0, 5.0] and observe the effect on convergence speed.

---

**Week 4 Complete.**

Week 4 Revision post coming up next — 5 MNC interview questions, 3 practice projects combining Days 25-31, and a community challenge.

**Week 5 starts with NLP and LLMs:**
Day 33: Tokenization and Embeddings — how text becomes numbers, what cosine similarity means, and why two sentences with no words in common can have nearly identical embeddings.